# Plasmid size and copy number: an explorer

<a href="https://colab.research.google.com/github/Azuromalachite/plasmid-ecoevo-lecture-2026/blob/main/reproductions/ramiro-scaling/explore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Quick start:** *Runtime &rarr; Run all*. Nothing needs editing; cell 0 holds everything that does.

Companion to [`RESULTS.md`](RESULTS.md) and [`REPORT.md`](REPORT.md).

Two datasets, both fetched and checked against a recorded SHA-256:

| | records | what it is |
|---|---:|---|
| **Ramiro-Martínez et al. 2025** | 3,961 | the curated *modelling* subset, PTU-labelled, no accessions |
| **Maddamsetti et al. 2025** | 12,006 plasmids | Supplementary Data 1, accession-keyed, includes copy numbers below 1 |

Panel B below is a **reproduction** of Maddamsetti's published figure on their own data, and it lands on their published numbers. Panel A imitates the form of the Shahzadi 2026 figure; Shahzadi reuses Maddamsetti's plasmids and copy-number estimates, so the two papers are not independent samples.

**Copy number in both files is modelled, not counted.** Neither ships an uncertainty. Nothing here should be read as resolving a difference smaller than an interval we cannot see.


In [ ]:
#@title 0. CONFIG — normal edits belong here
#@markdown Both datasets are fetched by URL and checked against a recorded
#@markdown SHA-256. A mismatch stops the run, because every number quoted in
#@markdown the markdown below refers to the recorded bytes.

# The curated modelling subset. `main` moves, so the hash is the only
# thing that makes a number in this notebook mean anything.
RAMIRO_URL = "https://raw.githubusercontent.com/PaulaRamiro/NpAUREO/main/Model/curateddataset.csv"  #@param {type:"string"}
RAMIRO_SHA = "23bee155b1c946bd565840c4cc0f5ac2e1fec6e4192578ba9f6752ee5f58f78f"  #@param {type:"string"}

# Maddamsetti Supplementary Data 1, from the publisher's static host.
# PMC serves the same file behind a "Preparing to download" page that an
# automated request cannot pass; this host does not.
MADDAMSETTI_URL = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-61205-2/MediaObjects/41467_2025_61205_MOESM3_ESM.csv"  #@param {type:"string"}
MADDAMSETTI_SHA = "131faa58f5528d814877cca9ffa8c3e56b41e92518385d740c1b7e530f2bc3a7"  #@param {type:"string"}

#@markdown Set only to look at an input that no longer matches. Every number
#@markdown in the prose then refers to a file you are not looking at.
EXPLORATORY = False  #@param {type:"boolean"}

#@markdown The published breakpoint, as a percentage of chromosome length.
#@markdown Held fixed by the adjusted model; see section 6.
BREAKPOINT_PERCENT = 1.7273321739634590  #@param {type:"number"}

#@markdown Smallest family that can carry its own intercept. Below three a
#@markdown family contributes almost nothing after centring.
MIN_FAMILY_SIZE = 3  #@param {type:"integer"}

# Matches reproduce.py. A coarser scan moves the knee by 0.01 percentage
# points and the upper slope in the third decimal, which would leave this
# notebook quietly disagreeing with the receipt it illustrates.
KNEE_GRID = 400  #@param {type:"integer"}

# Section 7 builds data with a known bend. Fixed so the figure a reader
# gets is the figure described in the text.
SEED = 11  #@param {type:"integer"}

#@markdown Plotly draws the interactive scatter. Colab ships it; off a
#@markdown hosted runtime the notebook falls back to a static matplotlib
#@markdown panel rather than failing.
INTERACTIVE_SCATTER = True  #@param {type:"boolean"}


In [ ]:
#@title 1. Fetch the Ramiro subset, and stop if it has moved
import hashlib, io, math, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize


def fetch(url, expected_sha, label):
    req = urllib.request.Request(url, headers={"User-Agent": "plasmid-ecoevo-repro/1.0"})
    raw = urllib.request.urlopen(req, timeout=300).read()
    got = hashlib.sha256(raw).hexdigest()
    print(f"{label}: {len(raw):,} bytes")
    print(f"  sha256 {got}")
    if got == expected_sha:
        print("  matches the recorded input")
    elif EXPLORATORY:
        print("  DIFFERS -- continuing because EXPLORATORY is True.")
        print("  Every number in the markdown refers to the other file.")
    else:
        raise SystemExit(
            f"{label}: input differs from the recorded bytes.\n"
            f"  expected {expected_sha}\n  got      {got}\n"
            "Set EXPLORATORY = True to look at it anyway.")
    return raw

ram_raw = fetch(RAMIRO_URL, RAMIRO_SHA, "Ramiro curated subset")


In [ ]:
#@title 2. Load Ramiro, and name the units trap
df = pd.read_csv(io.BytesIO(ram_raw), sep="\t", encoding="utf-8-sig")
df = df[(df.PCN > 0) & (df.Length_chr > 0)].copy()
df["plasmid_bp"] = 10 ** df.Logsize
df["rel_pct"] = 100 * df.plasmid_bp / df.Length_chr
df["x"] = np.log10(df.rel_pct)     # log10 PERCENT of chromosome
df["y"] = np.log10(df.PCN)

# UNITS. receipt.json stores breakpoint_log10 as log10 of a FRACTION
# (-1.7626). This notebook works in log10 of a PERCENT, so the same knee
# is log10(1.7273) = +0.2373. Mixing the two tests a knee a hundred
# times too small and silently returns a null; it has happened. The
# percentage prints below are the other half of the same trap: where x
# is already log10(percent), 10**c IS the percentage and multiplying
# by 100 again reports a bend at 184% of the chromosome.
KNEE_PCT = math.log10(BREAKPOINT_PERCENT)

print(f"{len(df):,} plasmids, {df.Predicted_ptu.nunique()} PTUs, "
      f"{df.Genus.nunique()} genera")
print(f"PCN {df.PCN.min():.3f} to {df.PCN.max():.2f}")
print()
print("Curation the authors document and the file does not show:")
print("  unassigned PTUs removed -- every record carries a named PTU")
print("  PCN outliers above 100 removed -- max here is 99.50")
print("  PCN below 1 removed -- min here is exactly 1.000")
print("The last is a one-sided truncation of the fitted quantity.")


## 1. The fitting machinery

`dAIC = AIC(segmented) - AIC(single line)`, so **negative favours the bend**. The single line costs two parameters, the hinge four when its knee is estimated and three when the knee is held fixed.


In [ ]:
#@title 3. Fits and model comparison
def ols(x, y):
    k, b = np.polyfit(x, y, 1)
    return k, b, float(((y - (k * x + b)) ** 2).sum())

def hinge(x, y, c):
    M = np.column_stack([np.ones_like(x), x, np.maximum(x - c, 0.0)])
    beta, *_ = np.linalg.lstsq(M, y, rcond=None)
    return float(((y - M @ beta) ** 2).sum()), beta

def best_hinge(x, y, n_grid=KNEE_GRID):
    grid = np.linspace(np.quantile(x, 0.05), np.quantile(x, 0.95), n_grid)
    sse, c = min((hinge(x, y, g)[0], g) for g in grid)
    _, beta = hinge(x, y, c)
    return c, float(beta[1]), float(beta[1] + beta[2]), sse

def aic(sse, n, k):
    return n * np.log(sse / n) + 2 * k

def compare(x, y):
    _, _, sse1 = ols(x, y)
    c, lo, hi, sse2 = best_hinge(x, y)
    n = len(x)
    return c, lo, hi, aic(sse2, n, 4) - aic(sse1, n, 2)

def demean(v, g):
    out = np.empty_like(v, dtype=float)
    for p in np.unique(g):
        m = g == p
        out[m] = v[m] - v[m].mean()
    return out


## 2. Panel A — the form of the Shahzadi figure

Copy number against plasmid length, coloured by host chromosome length. Shahzadi report an exponent near **−0.71**. Section 4 below recovers −0.717 as the single power law on Maddamsetti's data, which is the dataset Shahzadi reuse — so that number is not independent evidence about this relationship.


In [ ]:
#@title 4. Panel A — the Shahzadi form
fig, ax = plt.subplots(figsize=(6.4, 4.6), dpi=120)
sc = ax.scatter(df.Logsize, df.y, c=df.Length_chr, s=6, alpha=0.5,
                cmap="coolwarm", linewidths=0,
                norm=Normalize(*np.quantile(df.Length_chr, [0.02, 0.98])))
fig.colorbar(sc, ax=ax).set_label("chromosome length (bp)")
k, b, _ = ols(df.Logsize.values, df.y.values)
xs = np.linspace(df.Logsize.min(), df.Logsize.max(), 100)
ax.plot(xs, k * xs + b, color="black", lw=1.6)
ax.set_xlabel("plasmid length, log10 bp")
ax.set_ylabel("copy number, log10")
ax.set_title(f"Ramiro subset: PCN $\\propto$ size$^{{{k:.2f}}}$ (n={len(df):,})", fontsize=11)
ax.axhline(0, color="#888888", lw=0.8)
plt.tight_layout(); plt.show()
print(f"this sample {k:+.3f}   Shahzadi report -0.71 on a different, larger sample")


## 3. Maddamsetti's own data

Supplementary Data 1 from [10.1038/s41467-025-61205-2](https://doi.org/10.1038/s41467-025-61205-2), fetched from the publisher's static host. PMC serves supplementary files behind a "Preparing to download" page that an automated request cannot pass; this host does not.

The file carries `AnnotationAccession` and `SeqID`, so unlike the Ramiro modelling subset it can be joined to external annotation.


In [ ]:
#@title 5. Fetch Maddamsetti Supplementary Data 1
mad_raw = fetch(MADDAMSETTI_URL, MADDAMSETTI_SHA, "Maddamsetti Supplementary Data 1")
md_all = pd.read_csv(io.BytesIO(mad_raw), encoding="utf-8", encoding_errors="replace")
mp = md_all[md_all.SeqType.str.strip().str.lower() == "plasmid"].copy()
mp = mp[(mp.PIRACopyNumber > 0) & (mp.normalized_replicon_length > 0)]
mp["x"] = np.log10(100 * mp.normalized_replicon_length)   # percent
mp["y"] = np.log10(mp.PIRACopyNumber)

print(f"{len(md_all):,} replicons -> {len(mp):,} usable plasmids")
print(f"columns: {', '.join(list(md_all.columns)[:8])} ...")
print(f"plasmids with copy number below 1: {int((mp.y < 0).sum()):,} "
      f"({100*(mp.y < 0).mean():.1f}%)  <- the Ramiro file has none")


## 4. Panel B — reproducing their figure

Their published bend is at **1.84%** of chromosome length with slopes **−0.880** and **−0.125**. This is what their file gives.


In [ ]:
#@title 6. Panel B — reproducing their figure
c_m, lo_m, hi_m, d_m = compare(mp.x.values, mp.y.values)
k_m, b_m, _ = ols(mp.x.values, mp.y.values)

fig, ax = plt.subplots(figsize=(6.6, 4.6), dpi=120)
ax.scatter(mp.x, mp.y, s=4, alpha=0.35, linewidths=0, c="#56B4E9")
xs = np.linspace(mp.x.min(), mp.x.max(), 200)
_, beta = hinge(mp.x.values, mp.y.values, c_m)
ax.plot(xs, beta[0] + beta[1] * xs + beta[2] * np.maximum(xs - c_m, 0),
        color="#C1272D", lw=2)
ax.axhline(0, color="#888888", lw=0.8)
ax.set_xlabel("plasmid length as % of chromosome, log10")
ax.set_ylabel("copy number, log10")
ax.set_title(f"Maddamsetti Supplementary Data 1 (n={len(mp):,})", fontsize=11)
plt.tight_layout(); plt.show()

print(f"reproduced   bend {10**c_m:.2f}%  slopes {lo_m:+.3f} / {hi_m:+.3f}")
print(f"published    bend 1.84%      slopes -0.880 / -0.125")
print(f"single power law {k_m:+.3f}   (Shahzadi report -0.71)")


## 5. What our floor does to their data

`RESULTS.md` §3 notes that our lower slope is −0.683 where theirs is −0.880, and reasons that the PCN ≥ 1 floor cannot explain it because the floor barely touches small plasmids. That was an argument from the shape of our own data. Their file has the sub-1 plasmids ours lacks, so it can be tested directly.


In [ ]:
#@title 7. Apply our floor to their data
rows = []
for lab, sub in (("their data, all plasmids", mp),
                 ("their data, PCN >= 1 (our floor)", mp[mp.PIRACopyNumber >= 1])):
    c, lo, hi, _ = compare(sub.x.values, sub.y.values)
    rows.append((lab, len(sub), 10**c, lo, hi))
c_r, lo_r, hi_r, _ = compare(df.x.values, df.y.values)
rows.append(("Ramiro subset (ours), PCN >= 1", len(df), 10**c_r, lo_r, hi_r))

print(f"{'subset':<36}{'n':>7}{'bend':>9}{'below':>9}{'above':>9}")
for lab, n, c, lo, hi in rows:
    print(f"{lab:<36}{n:>7}{c:>8.2f}%{lo:>9.3f}{hi:>9.3f}")
gap = rows[0][3] - rows[2][3]
closed = rows[0][3] - rows[1][3]
print(f"\nthe floor closes {100*closed/gap:.0f}% of the lower-slope gap,")
print("and moves the bend and the upper slope toward ours as well.")


## 6. The explorer

Three model modes, because two of them answer different questions and an earlier version of this notebook conflated them.

| mode | what it asks | on this data |
|---|---|---|
| **pooled** | is there a bend at all, ignoring family? | −148.1 |
| **PTU-adjusted, knee fixed** | does the bend *at 1.73%* survive giving each family its own intercept? | −19.4 |
| **PTU-centred, knee re-fitted** | is there a bend *anywhere* inside families once their means are gone? | −1.3, and **uninformative** |

The middle one is the test that bears on the claim. The last one is the one to distrust: section 7 runs it on data built with a genuine shared threshold and it returns a *positive* −AIC, missing a bend that is really there. Two earlier versions of this notebook read its −1.3 as the bend disappearing, then as a different question answered no. It is a question this test cannot answer.


In [ ]:
#@title 8. Explorer — three model modes
import ipywidgets as W
from IPython.display import display

MODES = ["pooled",
         "PTU-adjusted, knee fixed at 1.73%",
         "PTU-centred, knee re-fitted"]

def explore(mode, genus, mob, floor, min_ptu):
    d = df
    if genus != "(all)": d = d[d.Genus == genus]
    if mob != "(all)": d = d[d.predicted_mobility.astype(str) == mob]
    d = d[d.PCN >= floor]
    if min_ptu > 1:
        vc = d.Predicted_ptu.value_counts()
        d = d[d.Predicted_ptu.isin(vc[vc >= min_ptu].index)]
    if len(d) < 40:
        print(f"only {len(d)} plasmids left -- too few to fit"); return

    x, y, g = d.x.values, d.y.values, d.Predicted_ptu.values
    note = ""
    if mode == MODES[0]:
        c, lo, hi, dd = compare(x, y)
        _, beta = hinge(x, y, c)
        knee = c
    else:
        vc = d.Predicted_ptu.value_counts()
        d2 = d[d.Predicted_ptu.isin(vc[vc >= MIN_FAMILY_SIZE].index)]
        if len(d2) < 40:
            print("too few plasmids in families of 3 or more"); return
        g2 = d2.Predicted_ptu.values
        yd = demean(d2.y.values, g2); xd = demean(d2.x.values, g2)
        n = len(xd); npt = d2.Predicted_ptu.nunique()
        if mode == MODES[1]:
            knee = KNEE_PCT
            straddle = sum(1 for p in np.unique(g2)
                           if (d2.x.values[g2 == p] < knee).any()
                           and (d2.x.values[g2 == p] >= knee).any())
            if straddle == 0:
                print("no family straddles the knee -- not identified"); return
            zd = demean(np.maximum(d2.x.values - knee, 0.0), g2)
            M1 = xd[:, None]; M2 = np.column_stack([xd, zd])
            b1, *_ = np.linalg.lstsq(M1, yd, rcond=None)
            b2, *_ = np.linalg.lstsq(M2, yd, rcond=None)
            s1 = float(((yd - M1 @ b1) ** 2).sum())
            s2 = float(((yd - M2 @ b2) ** 2).sum())
            dd = aic(s2, n, npt + 3) - aic(s1, n, npt + 2)
            lo, hi = b2[0], b2[0] + b2[1]
            note = f"{straddle} of {npt} families straddle the knee"
            x, y = d2.x.values, d2.y.values
            beta = None
        else:
            x, y = xd, yd
            c, lo, hi, dd = compare(x, y)
            _, beta = hinge(x, y, c); knee = c
            note = f"centred on {npt} PTUs; the knee is in centred coordinates"

    fig, ax = plt.subplots(figsize=(6.6, 4.2), dpi=110)
    ax.scatter(x, y, s=5, alpha=0.4, linewidths=0, c="#56B4E9")
    xs = np.linspace(x.min(), x.max(), 200)
    if beta is not None:
        ax.plot(xs, beta[0] + beta[1] * xs + beta[2] * np.maximum(xs - knee, 0),
                color="#C1272D", lw=2)
    else:
        ax.axvline(knee, color="#C1272D", lw=1.4, ls="--")
        ax.text(knee, ax.get_ylim()[1], "  knee held here", va="top",
                fontsize=8, color="#C1272D")
    ax.set_xlabel("log10 % of chromosome" + (" (family-centred)" if mode == MODES[2] else ""))
    ax.set_ylabel("log10 copy number" + (" (family-centred)" if mode == MODES[2] else ""))
    ax.set_title(f"{mode}   n={len(x):,}   dAIC {dd:+.1f}", fontsize=10)
    plt.tight_layout(); plt.show()

    print(f"slopes {lo:+.3f} then {hi:+.3f}   change {hi-lo:+.3f}")
    print(f"dAIC   {dd:+.1f}")
    if note: print(note)

display(W.interactive(
    explore,
    mode=W.Dropdown(options=MODES, value=MODES[0], description="model"),
    genus=W.Dropdown(options=["(all)"] + sorted(df.Genus.dropna().unique().tolist()),
                     value="(all)", description="genus"),
    mob=W.Dropdown(options=["(all)"] + sorted(df.predicted_mobility.dropna().astype(str).unique().tolist()),
                   value="(all)", description="mobility"),
    floor=W.FloatSlider(value=1.0, min=1.0, max=10.0, step=0.5,
                        description="PCN floor", continuous_update=False),
    min_ptu=W.IntSlider(value=1, min=1, max=50, step=1,
                        description="min n per PTU", continuous_update=False)))


## 7. Can the re-fitted test see a threshold that is really there?

This is the check that reversed the conclusion, so it is worth running rather than reading. Simulate 116 families in narrow size bands, each with its own intercept, all sharing one genuine bend at the fitted knee. A test worth trusting finds the bend it was given.


In [ ]:
#@title 9. Give the tests a threshold and see which find it
rng = np.random.default_rng(SEED)
delta_true = 0.533                 # the pooled slope change
sx, sy, sg = [], [], []
for j in range(116):
    band = rng.uniform(np.quantile(df.x, .02), np.quantile(df.x, .90))
    xf = band + rng.normal(0, 0.12, 34)
    sy.append(rng.normal(0.6, 0.45) - 0.683 * xf
              + delta_true * np.maximum(xf - KNEE_PCT, 0.0)
              + rng.normal(0, 0.25, 34))
    sx.append(xf); sg += [j] * 34
sx = np.concatenate(sx); sy = np.concatenate(sy); sg = np.array(sg)

def fe_daic(x, y, g, knee):
    """Fixed effects with the knee HELD. Returns (dAIC, slope change)."""
    yd, xd = demean(y, g), demean(x, g)
    zd = demean(np.maximum(x - knee, 0.0), g)
    npt = len(np.unique(g)); n = len(x)
    b1, *_ = np.linalg.lstsq(xd[:, None], yd, rcond=None)
    M2 = np.column_stack([xd, zd])
    b2, *_ = np.linalg.lstsq(M2, yd, rcond=None)
    s1 = float(((yd - xd * b1[0]) ** 2).sum())
    s2 = float(((yd - M2 @ b2) ** 2).sum())
    return aic(s2, n, npt + 3) - aic(s1, n, npt + 2), float(b2[1])

_, _, _, d_pool = compare(sx, sy)
_, _, _, d_rescan = compare(demean(sx, sg), demean(sy, sg))
d_fe, change = fe_daic(sx, sy, sg, KNEE_PCT)

print(f"simulated data built WITH a bend of +{delta_true:.3f} at 1.73%")
print()
print(f"  pooled                      dAIC {d_pool:+8.1f}")
print(f"  PTU-adjusted, knee fixed    dAIC {d_fe:+8.1f}   "
      f"recovers {change:+.3f} of +{delta_true:.3f}")
print(f"  PTU-centred, knee re-fitted dAIC {d_rescan:+8.1f}   <- misses it")
print()
print("A near-null from the re-fitted test therefore means nothing. The")
print("old power check planted breaks at a common CENTRED coordinate")
print("and recovered them, but a real threshold sits at a common")
print("ORIGINAL size, which centring scatters across families.")


## 8. Evidence is not magnitude

Adjusting for family takes ΔAIC from −148.1 to −19.4 — an 87% drop. That is a drop in *evidence*, and it is partly structural: the adjusted fit spends 116 parameters on intercepts while only 4.7% of the variation in size lies within families. The *magnitude* of the bend is the change in slope, and it falls far less.


In [ ]:
#@title 10. The two reductions measure different things
vc = df.Predicted_ptu.value_counts()
d2 = df[df.Predicted_ptu.isin(vc[vc >= MIN_FAMILY_SIZE].index)]
_, lo_p, hi_p, d_p = compare(df.x.values, df.y.values)
d_a, change_a = fe_daic(d2.x.values, d2.y.values,
                        d2.Predicted_ptu.values, KNEE_PCT)
pooled_change = hi_p - lo_p

print(f"{'':<10}{'dAIC':>10}{'slope change':>15}")
print(f"{'pooled':<10}{d_p:>10.1f}{pooled_change:>+15.3f}")
print(f"{'adjusted':<10}{d_a:>10.1f}{change_a:>+15.3f}")
print()
print(f"evidence  falls {100*(1-abs(d_a)/abs(d_p)):.0f}%")
print(f"magnitude falls {100*(1-change_a/pooled_change):.0f}%")
print()
print("Families account for roughly a third of the bend, not most of")
print("it. Reading the 87% as a share is the error the report and an")
print("earlier version of this notebook both made.")


## 11. The scatter itself

Everything above summarises this. Hover a point for its family, genus, mobility, size and copy number; drag to zoom; click a legend entry to drop a series. The fitted line refits to whatever is left after the copy-number floor.

The function below returns the figure and its caption and takes no notebook state, so it can be exercised without a kernel.


In [ ]:
#@title 11. The scatter — a pure function that returns it
try:
    import plotly.graph_objects as go
    HAVE_PLOTLY = INTERACTIVE_SCATTER
except ImportError:
    HAVE_PLOTLY = False

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#898781"]

def _frames():
    """Both datasets on one axis convention: log10 % of chromosome."""
    r = df.assign(dataset="Ramiro 2025",
                  family=df.Predicted_ptu, mobility=df.predicted_mobility,
                  genus=df.Genus)[["x", "y", "dataset", "family",
                                   "mobility", "genus", "PCN"]]
    m = mp.assign(dataset="Maddamsetti 2025", family="-",
                  mobility="-", genus=mp.Genus.fillna("-"),
                  PCN=mp.PIRACopyNumber)[["x", "y", "dataset", "family",
                                          "mobility", "genus", "PCN"]]
    return {"Ramiro 2025": r, "Maddamsetti 2025": m}

FRAMES = _frames()

def scatter_figure(dataset="Ramiro 2025", colour_by="mobility", floor=1.0):
    """Returns (figure_or_None, caption). Pure: no widgets, no display."""
    d = FRAMES[dataset]
    d = d[d.PCN >= floor]
    if len(d) < 40:
        return None, f"only {len(d)} plasmids left -- too few to fit"
    c, lo, hi, dd = compare(d.x.values, d.y.values)
    _, beta = hinge(d.x.values, d.y.values, c)
    cap = (f"{len(d):,} of {len(FRAMES[dataset]):,} plasmids. "
           f"bend {10**c:.2f}% of chromosome, slopes {lo:+.3f} then "
           f"{hi:+.3f}, dAIC {dd:+.1f}. Copy number is estimated from "
           f"sequencing coverage, not counted; no uncertainty is published.")
    if not HAVE_PLOTLY:
        return None, cap + "  (plotly unavailable: static panel above)"

    key = colour_by if colour_by in d.columns else None
    # Top three by size plus an explicit remainder. Taking the top four
    # and stopping drops every other point from the plot while the fit
    # still uses them, so the caption and the picture disagree.
    if key is None:
        groups = [(None, d)]
    else:
        ranked = sorted(d.groupby(key), key=lambda kv: -len(kv[1]))
        groups = ranked[:3]
        rest = [g for _, g in ranked[3:]]
        if rest:
            groups = groups + [("other", pd.concat(rest))]
    fig = go.Figure()
    for j, (name, g) in enumerate(groups):
        fig.add_trace(go.Scattergl(
            x=g.x, y=g.y, mode="markers", name=str(name or dataset),
            marker=dict(size=3.5, opacity=0.45, color=PALETTE[j % 4]),
            customdata=g[["family", "genus", "mobility", "PCN"]].values,
            hovertemplate=("%{customdata[1]} - %{customdata[0]}<br>"
                           "%{customdata[2]}<br>"
                           "%{x:.3f} log10 %% of chromosome<br>"
                           "%{customdata[3]:.2f} copies<extra></extra>")))
    xs = np.linspace(d.x.min(), d.x.max(), 200)
    fig.add_trace(go.Scatter(
        x=xs, y=beta[0] + beta[1]*xs + beta[2]*np.maximum(xs - c, 0),
        mode="lines", name="fitted", line=dict(color="#C1272D", width=2),
        hoverinfo="skip"))
    fig.add_hline(y=0, line_width=1, line_color="#898781",
                  annotation_text="one copy", annotation_position="top right")
    fig.update_layout(
        height=460, template="simple_white",
        margin=dict(l=60, r=20, t=30, b=55),
        xaxis_title="plasmid length, log10 % of chromosome",
        yaxis_title="copy number, log10",
        legend=dict(orientation="h", y=-0.18))
    return fig, cap

_f, _c = scatter_figure()
print(_c)


In [ ]:
#@title 12. The scatter — controls
import ipywidgets as W2
from IPython.display import display as _display

def _show(dataset, colour_by, floor):
    fig, cap = scatter_figure(dataset, colour_by, floor)
    if fig is not None:
        fig.show()
    print(cap)

_display(W2.interactive(
    _show,
    dataset=W2.Dropdown(options=list(FRAMES), value="Ramiro 2025",
                        description="dataset"),
    colour_by=W2.Dropdown(options=["mobility", "genus", "family", "dataset"],
                          value="mobility", description="colour by"),
    floor=W2.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.5,
                         description="PCN floor", continuous_update=False)))


---

### What this notebook does not establish

- That the residual bend is biological. It survives PTU adjustment but not restricting to *Escherichia* (+0.6) or raising the copy-number floor to three (−0.1), and these data cannot separate it from within-lineage curvature, PTU misclassification or sampling bias.
- Anything about inheritance from the 301-fold figure in the report. That evaluates a convex function at two medians; a group's loss rate is the average of the function over its distribution, which by Jensen's inequality is larger.
- That a PTU is a replication architecture. PTUs group plasmids by whole-sequence relatedness; shared PTU is not proof of shared replication control.
- That Shahzadi and Maddamsetti agree independently. Shahzadi reuse Maddamsetti's plasmids and copy-number estimates.
- Any uncertainty on a copy number. Neither source publishes one.
- That records are independent. Plasmids within a PTU are related by descent, so every AIC comparison here flatters itself.
